# Silver Layer – Cleaned & Deduplicated Tables

For each bronze table:
1. Read from `data/bronze/`
2. Drop nulls on business-critical columns (varies per table)
3. Deduplicate
4. Normalize text (lowercase/trim) and fix types
5. Write to `data/silver/`

In [1]:
from spark_windows import configure_windows_hadoop

configure_windows_hadoop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[4]")
    .appName("Silver Layer")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

c:\Users\yahya\Desktop\DE\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
BRONZE = "./data/bronze"
SILVER = "./data/silver"


def read_bronze(name):
    return spark.read.parquet(f"{BRONZE}/{name}.parquet")


def write_silver(df, name):
    df.write.mode("overwrite").parquet(f"{SILVER}/{name}.parquet")
    print(f"  ✓ {name}: {df.count()} rows written to silver")


def normalize_string_cols(df):
    """Trim and lowercase all string columns."""
    for field in df.schema.fields:
        if str(field.dataType) == "StringType()":
            df = df.withColumn(field.name, F.lower(F.trim(F.col(field.name))))
    return df


def summary(df, name):
    """Print row count and null counts per column."""
    print(f"\n--- {name} ---")
    print(f"Rows: {df.count()}")
    null_counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    )
    null_counts.show(truncate=False)

## 1. Orders

In [4]:
df = read_bronze("olist_orders_dataset")
summary(df, "orders (bronze)")

# Business logic: order_id and customer_id must exist; purchase timestamp is mandatory.
# Delivery dates can be null for non-delivered orders, so we only drop rows where
# delivered orders are missing their delivery timestamps.
df = df.dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])

# For delivered orders, drop rows missing any delivery date
delivered_with_nulls = (
    (F.col("order_status") == "delivered")
    & (
        F.col("order_approved_at").isNull()
        | F.col("order_delivered_carrier_date").isNull()
        | F.col("order_delivered_customer_date").isNull()
    )
)
df = df.filter(~delivered_with_nulls)

# Deduplicate on primary key
df = df.dropDuplicates(["order_id"])

# Normalize
df = normalize_string_cols(df)

summary(df, "orders (silver)")
write_silver(df, "olist_orders_dataset")


--- orders (bronze) ---
Rows: 99441
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


--- orders (silver) ---
Rows: 99418
+--------+-----------+------------+------------------------+-----------------+-------

## 2. Order Items

In [5]:
df = read_bronze("olist_order_items_dataset")
summary(df, "order_items (bronze)")

# All columns are critical for a line-item record
df = df.dropna(subset=["order_id", "order_item_id", "product_id", "seller_id", "price"])

# Deduplicate on composite key
df = df.dropDuplicates(["order_id", "order_item_id"])

# Normalize
df = normalize_string_cols(df)

summary(df, "order_items (silver)")
write_silver(df, "olist_order_items_dataset")


--- order_items (bronze) ---
Rows: 112650
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|0       |0            |0         |0        |0                  |0    |0            |
+--------+-------------+----------+---------+-------------------+-----+-------------+


--- order_items (silver) ---
Rows: 112650
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|0       |0            |0         |0        |0                  |0    |0            |
+--------+-------------+----------+---------+-------------------+-----+-------------+

  ✓ olist_order_items_dataset: 112650 rows written t

## 3. Customers

In [6]:
df = read_bronze("olist_customers_dataset")
summary(df, "customers (bronze)")

# customer_id is the PK, customer_unique_id links across orders
df = df.dropna(subset=["customer_id", "customer_unique_id"])

# Deduplicate on PK
df = df.dropDuplicates(["customer_id"])

# Normalize city names
df = normalize_string_cols(df)

summary(df, "customers (silver)")
write_silver(df, "olist_customers_dataset")


--- customers (bronze) ---
Rows: 99441
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|0          |0                 |0                       |0            |0             |
+-----------+------------------+------------------------+-------------+--------------+


--- customers (silver) ---
Rows: 99441
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|0          |0                 |0                       |0            |0             |
+-----------+------------------+------------------------+-------------+--------------+

  ✓ olist_customers_dataset: 99441 rows written 

## 4. Products (joined with category translation)

In [7]:
df_products = read_bronze("olist_products_dataset")
df_translation = read_bronze("product_category_name_translation")
summary(df_products, "products (bronze)")

# product_id is required
df_products = df_products.dropna(subset=["product_id"])

# Deduplicate on PK
df_products = df_products.dropDuplicates(["product_id"])

# Join category translation so products have English names
df_products = normalize_string_cols(df_products)
df_translation = normalize_string_cols(df_translation)

df = df_products.join(
    df_translation, on="product_category_name", how="left"
)

summary(df, "products (silver)")
write_silver(df, "olist_products_dataset")


--- products (bronze) ---
Rows: 32951
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0         |610                  |610                |610                       |610               |2               |2                |2                |2               |
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+


--- products (silver) ---
Rows: 32951
+---------------------+----------+-------------------+------------

## 5. Sellers

In [8]:
df = read_bronze("olist_sellers_dataset")
summary(df, "sellers (bronze)")

df = df.dropna(subset=["seller_id"])
df = df.dropDuplicates(["seller_id"])
df = normalize_string_cols(df)

summary(df, "sellers (silver)")
write_silver(df, "olist_sellers_dataset")


--- sellers (bronze) ---
Rows: 3095
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|0        |0                     |0          |0           |
+---------+----------------------+-----------+------------+


--- sellers (silver) ---
Rows: 3095
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|0        |0                     |0          |0           |
+---------+----------------------+-----------+------------+

  ✓ olist_sellers_dataset: 3095 rows written to silver


## 6. Payments

In [9]:
df = read_bronze("olist_order_payments_dataset")
summary(df, "payments (bronze)")

# order_id + payment_sequential is the composite key
df = df.dropna(subset=["order_id", "payment_sequential", "payment_value"])
df = df.dropDuplicates(["order_id", "payment_sequential"])
df = normalize_string_cols(df)

summary(df, "payments (silver)")
write_silver(df, "olist_order_payments_dataset")


--- payments (bronze) ---
Rows: 103886
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|0       |0                 |0           |0                   |0            |
+--------+------------------+------------+--------------------+-------------+


--- payments (silver) ---
Rows: 103886
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|0       |0                 |0           |0                   |0            |
+--------+------------------+------------+--------------------+-------------+

  ✓ olist_order_payments_dataset: 103886 rows written to silver


## 7. Reviews

In [10]:
df = read_bronze("olist_order_reviews_dataset")
summary(df, "reviews (bronze)")

# review_id + order_id is the key; score is required, comment can be null
df = df.dropna(subset=["review_id", "order_id", "review_score"])
df = df.dropDuplicates(["review_id", "order_id"])
df = normalize_string_cols(df)

summary(df, "reviews (silver)")
write_silver(df, "olist_order_reviews_dataset")


--- reviews (bronze) ---
Rows: 104162
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|1        |2236    |2380        |92157               |63079                 |8764                |8785                   |
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+


--- reviews (silver) ---
Rows: 101746
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+------

## 8. Geolocation

In [11]:
df = read_bronze("olist_geolocation_dataset")
summary(df, "geolocation (bronze)")

# zip + lat + lng must exist
df = df.dropna(subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])

# This table has massive duplicates – deduplicate on all columns
df = df.dropDuplicates()

df = normalize_string_cols(df)

summary(df, "geolocation (silver)")
write_silver(df, "olist_geolocation_dataset")


--- geolocation (bronze) ---
Rows: 1000163
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|0                          |0              |0              |0               |0                |
+---------------------------+---------------+---------------+----------------+-----------------+


--- geolocation (silver) ---
Rows: 738332
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|0                          |0              |0              |0               |0                |
+---------------------------+----------

## Done

All silver tables are written to `data/silver/`.